# 01. Data Analysis: EMBER 2018 v2 Static PE Malware Dataset

## Project Overview
This project builds a **static Windows Portable Executable (PE) malware classifier** using the **EMBER 2018 v2 dataset**.
Static analysis inspects file structure, byte entropy, imports, exports, and section characteristics **without executing the binary**, ensuring complete safety.

---
### Notebook Objectives
1. **Environment Verification**: Validate Python, NumPy, Pandas, scikit-learn, and EMBER versions.
2. **Dataset Verification**: Check local raw JSONL files, file sizes, and memory-mapped `.dat` representations.
3. **Load Vectorized Features**: Safely load memory-mapped feature arrays (`X_train`, `y_train`, `X_test`, `y_test`).
4. **Dataset Dimensions & Shapes**: Inspect feature dimensions (2381 static features per sample).
5. **Class Balance & Label Distribution**: Visualize benign (0) vs. malicious (1) sample distribution.
6. **Feature Quality Checks**: Verify absence of NaNs, infinite values, and validate label boundaries.


## A. Environment Verification
Verify that the runtime environment has all required libraries correctly configured.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import ember

print(f"Python Version:       {sys.version.split()[0]}")
print(f"NumPy Version:        {np.__version__}")
print(f"Pandas Version:       {pd.__version__}")
print(f"scikit-learn Version: {sklearn.__version__}")
print("EMBER Package:        Imported successfully!")


## B. Dataset Verification
Verify raw EMBER JSONL files and vectorized memory-mapped binary files.

In [ ]:
DATASET_DIR = os.path.abspath(os.path.join("..", "dataset", "ember2018"))
print(f"Dataset Path: {DATASET_DIR}")
print(f"Exists: {os.path.exists(DATASET_DIR)}")

# Inspect dataset files and sizes
files_info = []
for fname in sorted(os.listdir(DATASET_DIR)):
    fpath = os.path.join(DATASET_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        files_info.append({"Filename": fname, "Size (MB)": round(size_mb, 2)})

df_files = pd.DataFrame(files_info)
display(df_files)


## C & D. Load EMBER Vectorized Features & Inspect Shapes
We load features using `ember.read_vectorized_features()`.
This loads memory-mapped NumPy arrays from disk (`X_train.dat`, `y_train.dat`, `X_test.dat`, `y_test.dat`), avoiding memory exhaustion.


In [ ]:
X_train, y_train, X_test, y_test = ember.read_vectorized_features(DATASET_DIR)

print("Dataset Shapes and Memory Layout:")
print(f"  X_train: shape = {X_train.shape}, dtype = {X_train.dtype}")
print(f"  y_train: shape = {y_train.shape}, dtype = {y_train.dtype}")
print(f"  X_test:  shape = {X_test.shape}, dtype = {X_test.dtype}")
print(f"  y_test:  shape = {y_test.shape}, dtype = {y_test.dtype}")
print(f"\nStatic Features per Sample: {X_train.shape[1]}")


## E. Label Distribution & Class Balance
Analyze class distribution: `0 = Benign`, `1 = Malicious`.

In [ ]:
train_unique, train_counts = np.unique(y_train, return_counts=True)
test_unique, test_counts = np.unique(y_test, return_counts=True)

dist_df = pd.DataFrame({
    "Class": ["Benign (0)", "Malicious (1)"],
    "Train Count": train_counts,
    "Train %": (train_counts / len(y_train)) * 100,
    "Test Count": test_counts,
    "Test %": (test_counts / len(y_test)) * 100
})
display(dist_df)

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(x=["Benign (0)", "Malicious (1)"], y=train_counts, ax=axes[0], palette=["#2ecc71", "#e74c3c"])
axes[0].set_title(f"Train Set Distribution (Total: {len(y_train):,})")
axes[0].set_ylabel("Number of Samples")
for i, count in enumerate(train_counts):
    axes[0].text(i, count / 2, f"{count:,}\n({count/len(y_train)*100:.1f}%)", ha='center', va='center', color='white', fontweight='bold')

sns.barplot(x=["Benign (0)", "Malicious (1)"], y=test_counts, ax=axes[1], palette=["#2ecc71", "#e74c3c"])
axes[1].set_title(f"Test Set Distribution (Total: {len(y_test):,})")
axes[1].set_ylabel("Number of Samples")
for i, count in enumerate(test_counts):
    axes[1].text(i, count / 2, f"{count:,}\n({count/len(y_test)*100:.1f}%)", ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()


## F. Basic Feature Analysis & Sparsity
Check sparsity and summary statistics of EMBER feature representations.

In [ ]:
# Compute sparsity on a representative subset of training features
sample_subset = X_train[:1000]
zero_elements = np.count_nonzero(sample_subset == 0)
total_elements = sample_subset.size
sparsity = (zero_elements / total_elements) * 100

print(f"Feature Dimension: {X_train.shape[1]}")
print(f"Estimated Feature Sparsity: {sparsity:.2f}% (zeros across features)")

# Inspect summary statistics for the first 8 features
df_sample_features = pd.DataFrame(X_train[:1000, :8], columns=[f"Feature_{i}" for i in range(8)])
display(df_sample_features.describe().round(2))


## G. Data Quality Checks
Ensure data integrity: check for NaNs, Infs, and invalid labels.

In [ ]:
# Check for NaN and Infinite values
nan_train = np.isnan(X_train).any()
inf_train = np.isinf(X_train).any()
nan_test = np.isnan(X_test).any()
inf_test = np.isinf(X_test).any()

# Check label validity
valid_labels = set([0.0, 1.0])
train_labels_valid = set(np.unique(y_train)).issubset(valid_labels)
test_labels_valid = set(np.unique(y_test)).issubset(valid_labels)

print("Data Quality Verification Results:")
print(f"  - X_train contains NaN:      {nan_train}")
print(f"  - X_train contains Infinity: {inf_train}")
print(f"  - X_test contains NaN:       {nan_test}")
print(f"  - X_test contains Infinity:  {inf_test}")
print(f"  - y_train labels valid:      {train_labels_valid} (Labels: {set(np.unique(y_train))})")
print(f"  - y_test labels valid:       {test_labels_valid} (Labels: {set(np.unique(y_test))})")

assert not nan_train and not inf_train, "Data quality check failed: NaNs or Infs in X_train"
assert not nan_test and not inf_test, "Data quality check failed: NaNs or Infs in X_test"
print("\n[+] ALL DATA QUALITY CHECKS PASSED SUCCESSFULLY!")
